# Robinsons and Catlins River Mouth: Low-Tide Exposure

For a selected year, process Sentinel-2 scenes with less than 10% catalogue cloud cover at both sites. For each scene, record the nearest predicted low-tide time, height, and signed time offset; estimate whole-site dry area from SCL vegetation (class 4) and not-vegetated (class 5) pixels. Save per-site CSVs, RGB/SCL galleries, and dry-area-versus-low-tide scatter plots in each site's output folder.

In [16]:
import base64
import calendar
import html as html_module
import io
import os
import pathlib
import sys

import dotenv
import geopandas
import matplotlib.pyplot as plt
import numpy
import odc.stac
import pandas
import planetary_computer
import plotly.express as px
import rasterio.features
import requests
import xarray
import datetime
import zoneinfo
from IPython.display import FileLink, display

working_path = pathlib.Path.cwd().resolve()
module_path = working_path / "scripts" if (working_path / "scripts").is_dir() else working_path.parent / "scripts"
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

import sentinel2
import utils

%load_ext autoreload
%autoreload 2

In [79]:
data_path = utils.get_data_path()
site_polygons_path = data_path / "site_polygons"
output_root = data_path / "analysis" / "two_site_test"
output_root.mkdir(parents=True, exist_ok=True)

dotenv.load_dotenv()
tide_api_key = os.environ.get("TIDE_API")
if not tide_api_key:
    raise ValueError("Set TIDE_API in the environment or project .env file before querying tides.")

## Annual Site and Cloud Settings

Set the year and cloud-cover cutoff below. All matching scenes are retained; low-tide timing is recorded for comparison, not used to exclude scenes.

In [80]:
SITES = ("Robinsons", "CatlinsRiverMouth")
SITE_POLYGON_FILES = {
    site_name: site_polygons_path / f"{site_name}_polygon.gpkg"
    for site_name in SITES
}
SITE_OUTPUT_PATHS = {
    site_name: output_root / site_name
    for site_name in SITES
}

for site_name, polygon_file in SITE_POLYGON_FILES.items():
    if not polygon_file.is_file():
        raise FileNotFoundError(f"Missing site polygon for {site_name}: {polygon_file}")
    SITE_OUTPUT_PATHS[site_name].mkdir(parents=True, exist_ok=True)

print(f"Sites: {', '.join(SITES)}")
print(f"Per-site output root: {output_root}")

Sites: Robinsons, CatlinsRiverMouth
Per-site output root: C:\Local\repos\seagrass-detection\data\analysis\two_site_test


In [3]:
YEAR = 2025
MAX_ITEMS = 400
MAX_CLOUD_COVER_PCT = 10.0
TIDE_SEARCH_RADIUS_HRS = 12

print(f"Year: {YEAR}; STAC item cap per site: {MAX_ITEMS}")
print(f"Scene cloud-cover requirement: <{MAX_CLOUD_COVER_PCT:g}%")
print("Dry area: SCL 4 + 5 pixels divided by all rasterized site pixels")

Year: 2025; STAC item cap per site: 400
Scene cloud-cover requirement: <10%
Dry area: SCL 4 + 5 pixels divided by all rasterized site pixels


## Process Cloud-Filtered Scenes

For each site, retain every annual Sentinel-2 scene with catalogue cloud cover below 10%. Low tide is recorded for context but does not filter scenes. Dry area counts SCL 4+5 pixels against the full rasterized site polygon.

In [91]:
def get_low_tide_info(satellite_datetime_utc, lat, lon):
    """Returns the low tide information for the given satellite image and
      location - low tide height, time and time between the low tide and
      satellite image acquisition.
      lat: Latitude of the location in NZGD49
      lon: Longitude of the location in NZGD49
      """

    dotenv.load_dotenv()
    tide_api_key = os.environ.get("TIDE_API", None)
    if tide_api_key is None:
        raise ValueError("TIDE_API environment variable not set in .env file")

    satellite_time_nz_dt = satellite_datetime_utc.astimezone(
        zoneinfo.ZoneInfo('Pacific/Auckland'))
    start_date = ( # get the date 7hrs before the NZDT satellite time
        satellite_time_nz_dt - datetime.timedelta(hours=7)
        ).strftime(sentinel2.DATE_FORMAT_YYYYMMDD)
    tide_url = ( # Get tide for at least one low tide cyle either side
        f"{sentinel2.TIDE_API_STUB}?lat={lat}&long={lon}&datum=MSL"
        f"&numberOfDays=2&apikey={tide_api_key}&startDate={start_date}"
    )

    tide_query = requests.get(tide_url)
    tide_query.raise_for_status()
    
    tides = pandas.DataFrame.from_records(tide_query.json()["values"])
    tides["time"] = pandas.to_datetime(tides["time"], utc=True)
    tides = tides[tides["value"] < 0] # Low tides only

    time_from_low_tide = (tides["time"] - satellite_datetime_utc).abs().min()
    nearest_low = tides.loc[(tides["time"] - satellite_datetime_utc).abs().idxmin()]

    tide_info = { 
        "nearest_low_tide_height_msl": nearest_low["value"],
        "nearest_low_tide_time_utc": nearest_low["time"],
        "time_from_low_tide": format_tide_duration(time_from_low_tide),
        "hours_from_low_tide": time_from_low_tide.total_seconds() / 3600,
            }

    return tide_info

In [93]:
def format_tide_duration(time_from_low_tide):
    hours_from_low = time_from_low_tide.total_seconds() / 3600
    total_minutes = int(round(abs(float(hours_from_low)) * 60))
    hours, minutes = divmod(total_minutes, 60)
    direction = "at" if total_minutes == 0 else "before" if hours_from_low < 0 else "after"
    return f"{hours}h {minutes:02d}m {direction}"

In [ ]:
scl_classes = [
    (0, "No data", "#000000"),
    (1, "Saturated/defective", "#ff0000"),
    (2, "Dark area", "#2f2f2f"),
    (3, "Cloud shadow", "#643200"),
    (4, "Vegetation", "#00a000"),
    (5, "Not vegetated", "#ffe65a"),
    (6, "Water", "#0000ff"),
    (7, "Unclassified", "#808080"),
    (8, "Cloud, medium probability", "#c0c0c0"),
    (9, "Cloud, high probability", "#ffffff"),
    (10, "Thin cirrus", "#64c8ff"),
    (11, "Snow/ice", "#ff96ff"),
]
scl_legend_html = '<div class="legend">' + "".join(
    f'<span class="legend-item"><i style="background:{color}"></i>{code}: '
    f'{html_module.escape(label)}</span>'
    for code, label, color in scl_classes
) + "</div>"


def make_gallery(site_name, gallery_title, cards, output_file, legend_html=""):
    gallery_style = """
:root { color-scheme: light; font-family: "Segoe UI", sans-serif; color: #202a33; background: #f3f5f6; }
body { margin: 0; padding: 20px; }
header, .gallery, .legend { max-width: 1500px; margin: 0 auto 16px; }
h1 { margin: 0 0 6px; font-size: 22px; }
p { margin: 0; color: #56616b; }
.gallery { display: grid; grid-template-columns: repeat(auto-fill, minmax(min(320px, 100%), 1fr)); gap: 14px; }
.scene { min-width: 0; overflow: hidden; background: white; border: 1px solid #d5dce1; border-radius: 5px; }
.scene h2 { margin: 10px 12px 0; font-size: 15px; }
.scene img { display: block; width: 100%; aspect-ratio: 4 / 3; object-fit: contain; background: #f7f8f8; }
.caption { padding: 8px 12px 11px; font-size: 12px; line-height: 1.4; }
.legend { display: flex; flex-wrap: wrap; gap: 6px 12px; padding: 10px; background: white; border: 1px solid #d5dce1; }
.legend-item { display: inline-flex; align-items: center; gap: 5px; font-size: 11px; }
.legend-item i { display: inline-block; width: 14px; height: 14px; border: 1px solid #7b858c; }
@media (max-width: 600px) { body { padding: 10px; } }
"""
    page = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{html_module.escape(site_name)} {YEAR} {html_module.escape(gallery_title)}</title>
<style>{gallery_style}</style>
</head>
<body>
<header><h1>{html_module.escape(site_name)}: {YEAR} {html_module.escape(gallery_title)}</h1>
<p>{len(cards)} scenes with catalogue cloud cover under {MAX_CLOUD_COVER_PCT:g}%.</p></header>
{legend_html}
<main class="gallery">{''.join(cards)}</main>
</body>
</html>"""
    output_file.write_text(page, encoding="utf-8")
    print(f"  Saved {output_file}")

In [ ]:

site_results = {}
for site_name in SITES:
    print(f"\n{site_name}: search {YEAR} Sentinel-2 scenes")
    site_polygon_file = SITE_POLYGON_FILES[site_name]
    site_output_path = SITE_OUTPUT_PATHS[site_name]
    site_polygon = geopandas.read_file(site_polygon_file)
    if site_polygon.crs is None or site_polygon.empty:
        raise ValueError(f"Site polygon must be non-empty and have a CRS: {site_polygon_file}")
    
    geometry_WSG = site_polygon.buffer(sentinel2.S2_RESOLUTION).to_crs(utils.CRS_WSG)
    search_results = sentinel2.stac_search_with_retry(
        url=sentinel2.CATALOGUE_URL,
        max_items=MAX_ITEMS,
        collections=[sentinel2.COLLECTION],
        bbox=geometry_WSG.total_bounds,
        datetime=f"{YEAR}-01-01/{YEAR}-12-31",
        sortby=[{"field": "properties.eo:cloud_cover", "direction": "asc"}],
        get_collection=True,
    )

    cloud_filtered_items = []
    cloud_cover_by_date = {}
    for item in search_results.items:
        cloud_cover = item.properties.get("eo:cloud_cover")
        if cloud_cover is None or float(cloud_cover) >= MAX_CLOUD_COVER_PCT:
            continue
        acquisition_time = pandas.to_datetime(item.properties["datetime"], utc=True)
        scene_date = acquisition_time.strftime("%Y-%m-%d")
        cloud_cover = float(cloud_cover)
        cloud_filtered_items.append(item)
        cloud_cover_by_date[scene_date] = min(
            cloud_cover, cloud_cover_by_date.get(scene_date, cloud_cover)
        )

    print(f"  STAC items: {len(search_results.items)}; under cloud cutoff: {len(cloud_filtered_items)}")
    if not cloud_filtered_items:
        print(f"  No scenes below {MAX_CLOUD_COVER_PCT:g}% cloud cover")
        continue

    satellite_data = odc.stac.load(
        cloud_filtered_items,
        bbox=geometry_WSG.total_bounds,
        bands=sentinel2.RGB_BANDS + ["SCL"],
        chunks={},
        groupby="solar_day",
        resolution=sentinel2.S2_RESOLUTION,
        dtype="uint16",
        nodata=0,
        patch_url=planetary_computer.sign,
    )
    satellite_data = satellite_data.rio.clip(
        site_polygon.to_crs(satellite_data.rio.crs).geometry,
        all_touched=True,
        drop=True,
    ).load()
    if satellite_data.sizes.get("time", 0) == 0:
        print("  No cloud-filtered scenes intersect the site polygon")
        continue

    site_polygon_projected = site_polygon.to_crs(satellite_data.rio.crs)
    site_mask_values = rasterio.features.geometry_mask(
        [geometry.__geo_interface__ for geometry in site_polygon_projected.geometry],
        out_shape=(satellite_data.sizes["y"], satellite_data.sizes["x"]),
        transform=satellite_data.rio.transform(),
        invert=True,
        all_touched=True,
    )
    site_mask = xarray.DataArray(
        site_mask_values,
        coords={"y": satellite_data["y"], "x": satellite_data["x"]},
        dims=("y", "x"),
    )
    site_pixel_count = int(site_mask.sum())
    if site_pixel_count == 0:
        raise ValueError(f"No raster pixels fall inside {site_polygon_file}")

    scl_values = satellite_data["SCL"]
    dry_mask = scl_values.isin([4, 5]) & site_mask
    dry_pixel_counts = dry_mask.sum(dim=["x", "y"]).compute().values
    scl_covered_counts = ((scl_values > 0) & site_mask).sum(dim=["x", "y"]).compute().values
    scene_dates = satellite_data.time.dt.strftime("%Y-%m-%d").values.tolist()
    tide_cache = {}
    records = []
    # get tide info. Need lat/lon in NZGD49 and time in NZDT
    centroid_nzgd49 = site_polygon.to_crs(utils.CRS_NZGD49).geometry.union_all().centroid
    latitude_nzgd49, longitude_nzgd49 = float(centroid_nzgd49.y), float(centroid_nzgd49.x)
    for scene_index, scene_date in enumerate(scene_dates):
        satellite_datetime_utc = pandas.Timestamp(satellite_data.time.isel(time=scene_index).values)
        satellite_datetime_utc = (
            satellite_datetime_utc.tz_localize("UTC")
            if satellite_datetime_utc.tzinfo is None
            else satellite_datetime_utc.tz_convert("UTC")
        )
        tide_info = get_low_tide_info(satellite_datetime_utc, latitude_nzgd49, longitude_nzgd49)

        n_dry = int(dry_pixel_counts[scene_index])
        n_scl_covered = int(scl_covered_counts[scene_index])
        records.append({
            "site": site_name,
            "date": scene_date,
            "satellite_time_utc": satellite_datetime_utc,
            "cloud_cover_pct": cloud_cover_by_date.get(scene_date, numpy.nan),
            **tide_info,
            "dry_area_pct": 100 * n_dry / site_pixel_count,
            "scl_coverage_pct": 100 * n_scl_covered / site_pixel_count,
        })

    site_results[site_name] = pandas.DataFrame.from_records(records).sort_values("satellite_time_utc").reset_index(drop=True)
    site_results[site_name].to_csv(
        site_output_path / f"{site_name}_{YEAR}_low_tide_dry_area.csv", index=False
    )
    print(f"  Saved {len(site_results[site_name])} scene records to {site_output_path}")

    rgb_cards = []
    scl_cards = []
    for scene_index, row in site_results[site_name].iterrows():
        rgb = numpy.stack(
            [satellite_data[band].isel(time=scene_index).values for band in sentinel2.RGB_BANDS],
            axis=-1,
        ).astype(numpy.float32)
        rgb_valid = numpy.isfinite(rgb).all(axis=-1) & (rgb > 0).all(axis=-1)
        rgb_display = numpy.ones_like(rgb)
        if rgb_valid.any():
            for channel in range(3):
                pixels = rgb[..., channel][rgb_valid]
                low, high = numpy.percentile(pixels, [2, 98])
                if high > low:
                    rgb_display[..., channel] = numpy.clip(
                        (rgb[..., channel] - low) / (high - low), 0, 1
                    )
        rgb_display[~rgb_valid] = 1
        low_local = pandas.Timestamp(row["nearest_low_tide_time_utc"]).tz_convert("Pacific/Auckland")
        caption = (
            f"Dry {row['dry_area_pct']:.1f}% | Low {low_local:%d %b %H:%M} "
            f"({row['time_from_low_tide']}, {row['nearest_low_tide_height_msl']:.2f} m MSL)"
        )

        rgb_buffer = io.BytesIO()
        rgb_figure, rgb_axis = plt.subplots(figsize=(5, 3.75))
        rgb_axis.imshow(rgb_display)
        rgb_axis.axis("off")
        rgb_figure.savefig(
            rgb_buffer, format="jpg", dpi=90, bbox_inches="tight", pad_inches=0,
            pil_kwargs={"quality": 84, "optimize": True},
        )
        plt.close(rgb_figure)
        rgb_base64 = base64.b64encode(rgb_buffer.getvalue()).decode("ascii")
        safe_date = html_module.escape(str(row["date"]))
        rgb_cards.append(
            f'<article class="scene"><h2>{safe_date}</h2>'
            f'<img src="data:image/jpeg;base64,{rgb_base64}" alt="{site_name} RGB {safe_date}">'
            f'<div class="caption">{caption}</div></article>'
        )

        scl_array = satellite_data["SCL"].isel(time=scene_index).values
        scl_rgb = numpy.full((*scl_array.shape, 3), 238, dtype=numpy.uint8)
        for class_code, _, color in scl_classes:
            rgb_color = tuple(int(color[index:index + 2], 16) for index in (1, 3, 5))
            scl_rgb[(scl_array == class_code) & site_mask_values] = rgb_color
        scl_buffer = io.BytesIO()
        scl_figure, scl_axis = plt.subplots(figsize=(5, 3.75))
        scl_axis.imshow(scl_rgb)
        scl_axis.axis("off")
        scl_figure.savefig(
            scl_buffer, format="jpg", dpi=90, bbox_inches="tight", pad_inches=0,
            pil_kwargs={"quality": 90, "optimize": True},
        )
        plt.close(scl_figure)
        scl_base64 = base64.b64encode(scl_buffer.getvalue()).decode("ascii")
        scl_cards.append(
            f'<article class="scene"><h2>{safe_date}</h2>'
            f'<img src="data:image/jpeg;base64,{scl_base64}" alt="{site_name} SCL {safe_date}">'
            f'<div class="caption">{caption}</div></article>'
        )

    make_gallery(
        site_name, "RGB scenes", rgb_cards,
        site_output_path / f"{site_name}_{YEAR}_rgb.html",
    )
    make_gallery(
        site_name, "SCL scenes", scl_cards,
        site_output_path / f"{site_name}_{YEAR}_scl.html",
        scl_legend_html,
    )

    scatter_data = site_results[site_name].copy()
    scatter_data["date_color"] = pandas.to_datetime(
        scatter_data["satellite_time_utc"], utc=True
    ).astype("int64") / 1e9
    color_ticks = numpy.linspace(scatter_data["date_color"].min(), scatter_data["date_color"].max(), 5)
    color_tick_labels = pandas.to_datetime(color_ticks, unit="s", utc=True).strftime("%d %b").tolist()
    scatter_data["abs_hours_from_low_tide"] = scatter_data["hours_from_low_tide"].abs()
    scatter = px.scatter(
        scatter_data,
        x="abs_hours_from_low_tide",
        y="dry_area_pct",
        color="date_color",
        color_continuous_scale="Viridis",
        hover_name="date",
        hover_data={
            "time_from_low_tide": True,
            "nearest_low_tide_height_msl": ":.2f",
            "cloud_cover_pct": ":.1f",
            "scl_coverage_pct": ":.1f",
            "date_color": False,
        },
        labels={
            "abs_hours_from_low_tide": "Absolute duration from low tide (hours)",
            "dry_area_pct": "Dry area (SCL 4+5, % of site)",
            "date_color": "Acquisition date",
        },
        title=f"{site_name}: Dry area vs time from low tide ({YEAR})",
    )
    scatter.update_layout(
        height=650,
        hovermode="closest",
        coloraxis_colorbar={
            "title": "Acquisition date",
            "tickvals": color_ticks.tolist(),
            "ticktext": color_tick_labels,
        },
        yaxis={"range": [0, 100]},
    )
    scatter_file = site_output_path / f"{site_name}_{YEAR}_dry_vs_low_tide.html"
    scatter.write_html(scatter_file, include_plotlyjs=True, auto_open=False)
    print(f"  Saved scatter: {scatter_file}")
    display(site_results[site_name])
    display(FileLink(str(site_output_path / f"{site_name}_{YEAR}_rgb.html")))
    display(FileLink(str(site_output_path / f"{site_name}_{YEAR}_scl.html")))
    display(FileLink(str(scatter_file)))


Robinsons: search 2025 Sentinel-2 scenes
  STAC items: 140; under cloud cutoff: 27
